In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from langgraph.types import interrupt,Command
from langgraph.checkpoint.memory import InMemorySaver
from typing import Literal

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
class MailState(BaseModel):
    query: str = ""
    draft: str = ""
    human_feedback: str = ""
    final_response: str = ""

In [ ]:
###defining nodes
def DraftEmail(state: MailState) -> MailState:
    if state.human_feedback:
        draft=llm.invoke(f"""Rewrite the email for {state.query},
                         you have generated this draft mail: {state.draft},
                         Human Feedback to apply:{state.human_feedback}
                           """).content
    draft = llm.invoke(state.query).content
    state.draft = draft
    return state


def HumanFeedback(state: MailState) -> MailState:
    feedback = interrupt(
        {"draft_mail": state.draft, 
         "question": "do you want to continue or re-write the mail ?"} )
    fb =(feedback or "").strip().lower()
    if fb in ["continue", "yes","ok","proceed","approved"]:
        state.human_feedback = ""
        return state
    else:
        state.human_feedback = fb
        return state
    


def FinalizeNode(state: MailState) -> MailState:
    # Perform any final processing or validation here
    if state.human_feedback:
        state.final_response =  state.human_feedback
    else:
        print("Email sent successfully!")
        state.final_response = "Email sent successfully!"
    return state 


def conditional_routing(state: MailState) -> Literal["draft","final"]:
    if state.human_feedback:
        return "draft"
    else:
        return "final"   


In [ ]:
graph_builder = StateGraph(MailState)

#add nodes to the graph
graph_builder.add_node("draft", DraftEmail)
graph_builder.add_node("humanfb", HumanFeedback)
graph_builder.add_node("final", FinalizeNode)

## edge connections
graph_builder.add_edge(START, "draft")
graph_builder.add_edge("draft", "humanfb")
graph_builder.add_conditional_edges("humanfb", conditional_routing)
graph_builder.add_edge("final", END)

graph= graph_builder.compile(checkpointer=InMemorySaver())

In [ ]:
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())

In [ ]:
config = {"configurable":{"thread_id":"1"}}

In [ ]:
res = graph.invoke({"query":"Please draft an email to my manager requesting a day off on july 15."}, config=config)
print(res["draft"])

In [ ]:
res = graph.invoke(Command(resume ="change date to july 28th"), config=config)  
print(res["draft"]) 
